<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-01-logica/06%20-%20Quantificadores%20e%20Predicados%20em%20Redes%20de%20Sensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 06 - Motor de Varredura de Predicados em Redes de Sensores
## Célula de Manufatura Flexível (FMS)

Neste notebook implementamos a avaliação de predicados unários e binários e os quantificadores universais $\forall$ (`FORALL`) e existenciais $\exists$ (`EXISTS`) sobre a malha de sensores e atuadores distribuídos da manufatura flexível.

In [ ]:
from dataclasses import dataclass
from typing import List, Callable, Any

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass
class SensorProcesso:
    tag: str
    setor: str
    tipo: str
    valor: float
    unidade: str
    limite: float
    falha_comunicacao: bool = False

def FORALL(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return all(predicado(x) for x in dominio)

def EXISTS(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return any(predicado(x) for x in dominio)

rede_sensores = [
    # Setor 300: Loteamento e Caixas (Limite: 10 pecas)
    SensorProcesso('US-301', 'Setor 300', 'CONTAGEM', 10.0, 'pecas', 10.0),
    SensorProcesso('US-302', 'Setor 300', 'CONTAGEM', 4.0,  'pecas', 10.0),
    SensorProcesso('US-303', 'Setor 300', 'CONTAGEM', 7.0,  'pecas', 10.0),

    # Setor 200: Sensores de Inspeção de Cor (Sinal 0 a 10V)
    SensorProcesso('AS-201', 'Setor 200', 'COR_RGB',  9.2,  'V',     10.0),
    SensorProcesso('AS-202', 'Setor 200', 'COR_RGB',  0.5,  'V',     10.0),
    SensorProcesso('AS-203', 'Setor 200', 'COR_RGB',  0.3,  'V',     10.0),

    # Setor 100: Instrumentação de Alimentação
    SensorProcesso('ZS-101', 'Setor 100', 'POSICAO',  1.0,  'bool',  1.0),
    SensorProcesso('LS-101', 'Setor 100', 'NIVEL',    0.0,  'bool',  1.0)
]

sensores_contagem = [s for s in rede_sensores if s.tipo == 'CONTAGEM']

existe_caixa_cheia = EXISTS(sensores_contagem, lambda s: s.valor >= s.limite)
todos_comunicando = FORALL(rede_sensores, lambda s: not s.falha_comunicacao)

print(f"1. Existe Caixa Cheia / Lote Completo (EXISTS): {existe_caixa_cheia}")
print(f"2. Todos os Sensores Comunicando (FORALL): {todos_comunicando}")

tabela = [
    {
        "Tag": s.tag,
        "Setor": s.setor,
        "Valor": f"{s.valor} {s.unidade}",
        "Limite": f"{s.limite} {s.unidade}",
        "Alarme/Cheio": s.valor >= s.limite
    }
    for s in rede_sensores
]

print("\n" + formatar_tabela(tabela))

assert existe_caixa_cheia is True